# CSC 303: Academic Advisor AI Model

This Google Colab notebook trains an explainable k-nearest-neighbours academic-support classifier and exports it as a `.pkl` file for the Flask web application. All training records are synthetic demonstration data.

In [ ]:
# Run this cell in Google Colab. It creates the model class used by the web app.
import pickle
from dataclasses import dataclass
from math import sqrt

@dataclass
class AcademicAdvisorModel:
    k: int = 5
    training_data: list = None

    def fit(self, records):
        self.training_data = [dict(record) for record in records]
        return self

    @staticmethod
    def _normalise(student):
        return (float(student['gpa'])/5, float(student['failed_courses'])/6,
                float(student['attendance'])/100, float(student['hours_per_week'])/40,
                float(student['level'])/500)

    def predict_priority(self, student):
        target = self._normalise(student)
        def distance(record):
            return sqrt(sum((a-b)**2 for a, b in zip(target, self._normalise(record))))
        neighbours = sorted(self.training_data, key=distance)[:self.k]
        labels = [row['priority'] for row in neighbours]
        prediction = max(set(labels), key=labels.count)
        return prediction, round(100 * labels.count(prediction) / len(labels), 1)

In [ ]:
# Synthetic academic profiles: features and the known support-priority label.
training_data = [
 {'gpa':1.20,'failed_courses':4,'attendance':45,'hours_per_week':4,'level':200,'priority':'High'},
 {'gpa':1.55,'failed_courses':3,'attendance':62,'hours_per_week':6,'level':300,'priority':'High'},
 {'gpa':1.80,'failed_courses':2,'attendance':70,'hours_per_week':8,'level':100,'priority':'High'},
 {'gpa':2.10,'failed_courses':2,'attendance':68,'hours_per_week':7,'level':200,'priority':'High'},
 {'gpa':2.35,'failed_courses':1,'attendance':72,'hours_per_week':9,'level':300,'priority':'Medium'},
 {'gpa':2.50,'failed_courses':1,'attendance':76,'hours_per_week':12,'level':400,'priority':'Medium'},
 {'gpa':2.72,'failed_courses':0,'attendance':74,'hours_per_week':10,'level':200,'priority':'Medium'},
 {'gpa':2.90,'failed_courses':1,'attendance':82,'hours_per_week':14,'level':100,'priority':'Medium'},
 {'gpa':3.20,'failed_courses':0,'attendance':80,'hours_per_week':14,'level':300,'priority':'Low'},
 {'gpa':3.45,'failed_courses':0,'attendance':85,'hours_per_week':16,'level':400,'priority':'Low'},
 {'gpa':3.70,'failed_courses':0,'attendance':90,'hours_per_week':20,'level':200,'priority':'Low'},
 {'gpa':4.30,'failed_courses':0,'attendance':94,'hours_per_week':24,'level':500,'priority':'Low'}
]
model = AcademicAdvisorModel(k=5).fit(training_data)
model.predict_priority({'gpa': 2.4, 'failed_courses': 1, 'attendance': 75, 'hours_per_week': 11, 'level': 300})

In [ ]:
# Save and download the model. Put it in this project's models/ folder.
model_path = 'academic_advisor_model.pkl'
with open(model_path, 'wb') as file:
    pickle.dump(model, file)

from google.colab import files
files.download(model_path)

## Deployment note
The project repository contains `train_model.py`, which produces a compatible `.pkl` file with the same `AcademicAdvisorModel` definition. The Flask application loads `models/academic_advisor_model.pkl` and uses it to make predictions from the browser form.